# 判定の門の学習（コンパウンドIME 用）

「この変換で人工知能の生成（all_zenzai）が要るか」を決める、文字単位の小さな模型（約450万の数）を学習します。
入力は前の文・読み・生成なしの候補の上位、出力は「生成が要る確率」と候補ごとの点です。
材料（学習用の文の記録）は GitHub Actions が作った成果物 `jevgate-material` です（`docs/jev-gate.md`）。

**使い方**

1. 上のメニューの「ランタイム」→「ランタイムのタイプを変更」で **L4 GPU** を選んで「保存」を押します
   （T4 でも動きます。CPU でも動きますが遅いです）。
2. 下のセルを **上から順に1つずつ** 押します（セルの左の ▶ ボタン）。
3. 途中で接続が切れたら、もう一度 **①から順に** 押してください。終わった所は飛ばし、学習は最後の
   チェックポイント（10分ごとにドライブへ保存）から続きます。
4. 材料の取り方は⑤に2通りあります（GitHub から自動で取る／自分で取ってドライブに置く）。

かかる時間の目安（L4、30万文・4候補・8周）: 学習 約1〜2時間。⑦が始まると実際の速さで残り時間を出し直します。
**できた模型はまだ公開しないでください。** 試験の組の机上の評価（⑨）で、生成に戻すしきい値ごとの正解数を確かめてから決めます。

## ① 設定
ふつうは変えなくてかまいません。▶ を押してください。

In [ ]:
#@title ① 設定
RUN_NAME = "v1"            # 結果を置くフォルダの名前。別の設定で作り直すときは名前を変える
REPO_URL = "https://github.com/J-KAMMA/compound-ime.git"
REPO_REF = "exp/jev-gate"  # 使うリポジトリの枝または版
GH_REPO = "J-KAMMA/compound-ime"
MATERIAL_RUN = ""          # 材料を作った GitHub Actions の run の番号（⑤で自動で取るとき。空なら自分で置いた材料を使う）
MATERIAL_NAME = "m1"       # 材料を置くフォルダの名前（ドライブの jev-gate/material/<名前>）。run が違えば名前も変える

K = 4                      # 入力に入れる生成なしの候補の数（0〜16）。大きいほど正確だが推論が遅い（docs/jev-gate.md の時間の表）
D, LAYERS, HEADS, FFN = 256, 4, 4, 1024   # 模型の大きさ。d=256・4層で約450万、d=192・3層で約220万
EPOCHS = 8                 # 同じ記録を何回くり返すか
BATCH = 256                # 1歩の文の数
LEARNING_RATE = 6e-4
SEED = 20261019
CHECKPOINT_MINUTES = 10    # 途中の結果をドライブに残す間隔（分）
LIMIT = None               # 試しに少なく読むときの文の数（None なら全部）

DRIVE_ROOT = "/content/drive/MyDrive/jev-gate"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
MATERIAL_DIR = f"{DRIVE_ROOT}/material/{MATERIAL_NAME}"
WORK = "/content/work"
print("設定しました。結果の置き場所:", RUN_DIR)

## ② GPU の確認と Google ドライブへの接続
▶ を押すと、Google ドライブへの接続の許可を求める画面が出ます。**自分のアカウントを選んで許可**してください。

In [ ]:
#@title ② GPU の確認と Google ドライブへの接続
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPU。遅いです）")
from google.colab import drive
drive.mount("/content/drive")
for d in (RUN_DIR, MATERIAL_DIR, WORK):
    os.makedirs(d, exist_ok=True)
print("ドライブにつながりました:", RUN_DIR)

## ③ 台本を取ってくる
学習の台本（`tools/jev-gate/`）と評価の道具を GitHub から取ります。

In [ ]:
#@title ③ 台本を取ってくる
import os, subprocess, sys, importlib
REPO = f"{WORK}/compound-ime"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
KIT = f"{REPO}/tools/jev-gate"
sys.path.insert(0, KIT)
sys.path.insert(0, f"{REPO}/tools")
import jevgate as jg
importlib.reload(jg)
print("リポジトリの版:", REPO_COMMIT)
cfg = dict(k=K, d=D, layers=LAYERS, heads=HEADS, ffn=FFN)
m = jg.JevGate(4096, {**jg.DEFAULT_CFG, **cfg})
print(f"模型の数: 約{jg.count_params(m) / 1e6:.2f} 百万（語彙 4096 のとき）")
del m

## ④ 道具の確認
Colab には torch と numpy が入っているので、追加で入れるものはありません。

In [ ]:
#@title ④ 確認
import torch, numpy
print("torch", torch.__version__, " numpy", numpy.__version__)

## ⑤ 材料を取ってくる
GitHub Actions の成果物 `jevgate-records-<組の番号>`（学習用の記録）と `jevgate-test-records`（試験の組の記録）を
ドライブの `jev-gate/material/<MATERIAL_NAME>` に置きます。

- **自動で取る**: ①の `MATERIAL_RUN` に run の番号を入れ、Colab の左のカギのアイコン（シークレット）に
  `GH_TOKEN`（GitHub の細かい権限のトークン。リポジトリ J-KAMMA/compound-ime の「Actions: 読み取り」だけ）を
  登録して「ノートブックからのアクセス」をオンにします。
- **自分で置く**: GitHub の run の画面から成果物（zip）をダウンロードして展開し、`records-*.jsonl.gz` と
  `test-records.jsonl.gz` を `MATERIAL_DIR` に置きます（ブラウザからドライブへ）。

In [ ]:
#@title ⑤ 材料を取ってくる
import glob, io, json, os, urllib.request, zipfile
have = sorted(glob.glob(f"{MATERIAL_DIR}/records-*.jsonl.gz"))
if have and os.path.exists(f"{MATERIAL_DIR}/test-records.jsonl.gz"):
    print("材料はもうそろっています:", len(have), "組")
elif MATERIAL_RUN:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    def api(url):
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"})
        return urllib.request.urlopen(req)
    arts, page = [], 1
    while True:
        res = json.load(api(f"https://api.github.com/repos/{GH_REPO}/actions/runs/{MATERIAL_RUN}/artifacts?per_page=100&page={page}"))
        arts += res["artifacts"]
        if len(res["artifacts"]) < 100:
            break
        page += 1
    for a in arts:
        if not (a["name"].startswith("jevgate-records-") or a["name"] == "jevgate-test-records") or a["expired"]:
            continue
        data = api(a["archive_download_url"]).read()
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for n in z.namelist():
                if n.endswith(".jsonl.gz"):
                    open(f"{MATERIAL_DIR}/{os.path.basename(n)}", "wb").write(z.read(n))
        print("取りました:", a["name"], a["size_in_bytes"] // 1_000_000, "MB")
else:
    raise SystemExit(f"材料がありません。{MATERIAL_DIR} に records-*.jsonl.gz と test-records.jsonl.gz を置くか、①の MATERIAL_RUN を入れてください。")
RECORDS = sorted(glob.glob(f"{MATERIAL_DIR}/records-*.jsonl.gz"))
TEST_RECORDS = f"{MATERIAL_DIR}/test-records.jsonl.gz"
print("学習用の記録:", len(RECORDS), "ファイル。試験の組の記録:", os.path.exists(TEST_RECORDS))

## ⑥ 件数とラベルの割合
学習用の文の数、「生成が要る」ラベルの割合などを出します。

In [ ]:
#@title ⑥ 件数とラベルの割合
import subprocess
print(subprocess.run([sys.executable, f"{REPO}/tools/jev_gate_build.py", "stats", *RECORDS], capture_output=True, text=True).stdout)

## ⑦ 学習する（いちばん長い）
**途中で切れても、①から押し直せば続きから始まります。** チェックポイントは10分ごとにドライブへ保存されます。

In [ ]:
#@title ⑦ 学習する
import os
FINAL = f"{RUN_DIR}/run/final.pt"
if os.path.exists(FINAL):
    print("学習はもう終わっています:", FINAL)
else:
    jg.train(RECORDS, f"{RUN_DIR}/run", cfg, epochs=EPOCHS, batch_size=BATCH, lr=LEARNING_RATE, seed=SEED,
             limit=LIMIT, checkpoint_minutes=CHECKPOINT_MINUTES, cache_dir=f"{WORK}/cache")

## ⑧ 確かめ用の成績
学習に使っていない文（記録の id の指紋で2%）での、各頭の AUC と、生成に回す割合ごとの正解数です。

In [ ]:
#@title ⑧ 確かめ用の成績
import json
rows = [json.loads(l) for l in open(f"{RUN_DIR}/run/log.txt", encoding="utf-8") if l.startswith("{")]
for r in rows[-3:]:
    print("歩", r["step"])
    jg.print_eval(r["eval"])

## ⑨ 試験の組での机上の評価
`run 39249431055` の材料から作った試験の組の記録（学習に使っていない）に模型を当て、しきい値ごとに
「正解数」「生成に回る割合」「速さの見込み」を出します。**しきい値は⑧の確かめ用で決めてください。**
ここで試験の文を見て決めると本番より良く見えます。

In [ ]:
#@title ⑨ 試験の組での机上の評価
import subprocess, sys
PRED = f"{RUN_DIR}/result/pred-test.tsv"
os.makedirs(f"{RUN_DIR}/result", exist_ok=True)
jg.predict(f"{RUN_DIR}/run/final.pt", [TEST_RECORDS], PRED)
for select in ("student", "zenz"):
    out = subprocess.run([sys.executable, f"{KIT}/gate_eval.py", TEST_RECORDS, "--pred", PRED, "--head", "need_k",
                          "--select", select, "--by-set"], capture_output=True, text=True)
    print(out.stdout)
    open(f"{RUN_DIR}/result/eval-{select}.md", "w", encoding="utf-8").write(out.stdout)

## ⑩ 書き出し
C++ の推論用の重みの表（JEVG 形式、単精度と半精度）と語彙、C++ との答え合わせ用の見本を `result` に置きます。

In [ ]:
#@title ⑩ 書き出し
jg.export(f"{RUN_DIR}/run/final.pt", f"{RUN_DIR}/result/jevgate-fp32.bin")
jg.export(f"{RUN_DIR}/run/final.pt", f"{RUN_DIR}/result/jevgate-fp16.bin", fp16=True)
jg.dump_case(f"{RUN_DIR}/run/final.pt", [TEST_RECORDS], f"{RUN_DIR}/result/case.txt")
import shutil
shutil.copy(f"{RUN_DIR}/run/log.txt", f"{RUN_DIR}/result/train-log.txt")
print("できあがり。Google ドライブの次の場所にあります:", f"{RUN_DIR}/result")
print("次の手順: 手元で tools/jev-gate/jevgate_infer.cpp を組み立て、`jevgate_infer check jevgate-fp32.bin case.txt` で答え合わせ、")
print("`jevgate_infer bench --model jevgate-fp16.bin --k <K> --threads 4` で時間を測ります（docs/jev-gate.md）。")
print("※ この模型はまだ公開しないでください。")